<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "624"
iso_country = "Guinea-Bissau"
year = 2018
import_portfolio = "h5_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.550*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Guinea-Bissau': 

Number of trade records present in dataset: 5378
Number of partner countries present in dataset: 77
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$105.1m
Total weight within the dataset: 188.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,624,Guinea-Bissau,686,Senegal,Import,Sea,10515,"010515 | Poultry; live, guine",0105,"0105 | Poultry; live, fowls",01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,0.35600,NaN,0.35600,NaN,0.00,0.30,0.158384,398.0,247.305658,0.039169,['Container'],['Container'],0.000201,0.000200,0.000200,0.000200,0.000201,0.000201,0.00021,0.000373,0.000231,0.000215,0.000216,0.000221,0.000206,0.000365,0.000244,0.000223,0.000218,0.000228,0.000202,0.000397,0.000302,0.000265,0.000245,0.000238,0.000198,0.000422,0.000347,0.000325,0.000262,0.000255,0.000194,0.000428,0.000410,0.000413,0.000284,0.000268,0.000191,0.000419,0.000401,0.000439,0.000326,0.000273,0.000187,0.000374,0.000390,0.000408,0.000305,0.000276,0.000183,0.000343,0.000349,0.000433,0.000358,0.000278,nan
1,1,624,Guinea-Bissau,620,Portugal,Import,Sea,20120,020120 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,255.03213,NaN,255.03213,kg,138.55,138.55,38.280323,3360.0,2087.806560,79.921909,['Container'],['Container'],0.409718,0.408994,0.409065,0.409065,0.409389,0.409389,0.42865,0.760501,0.472236,0.439053,0.440256,0.451163,0.419695,0.744461,0.497408,0.455839,0.445314,0.465573,0.411841,0.810599,0.617076,0.539751,0.499269,0.485306,0.404132,0.861543,0.708851,0.663572,0.534457,0.520180,0.396569,0.873809,0.835604,0.843249,0.578818,0.546455,0.388854,0.855792,0.818965,0.896738,0.665474,0.556494,0.381586,0.762559,0.795467,0.833321,0.622913,0.563071,0.374175,0.700556,0.711868,0.883889,0.731102,0.566825,21111


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $44.5m (42.3%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
10 | Cereals               19128306.4  33240910.9     18.2     17.6
22 | Beverages, spirits a   7844503.9  18892804.3      7.5     10.0
11 | Products of the mill   5949833.6  14511088.4      5.7      7.7
17 | Sugars and sugar con   2550848.2   6089227.2      2.4      3.2
04 | Dairy produce; birds   2362197.6   1268593.6      2.2      0.7
02 | Meat and edible meat   1797814.3    969815.6      1.7      0.5
15 | Animal, vegetable or   1388043.2   1294580.4      1.3      0.7
20 | Preparations of vege   1116895.8    807596.4      1.1      0.4
19 | Preparations of cere    823996.5    369321.0      0.8      0.2
21 | Miscellaneous edible    575928.1    222484.4      0.5      0.1
07 | Vegetables and certa    529283.6    901312.1      0.5      0.5
16 | Meat, fish,

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $43.9m (41.7%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       30782544.4  51899204.4     29.3     27.5
Food & Beverages  13064806.4  26151712.7     12.4     13.9
Fishing              10241.4      2121.0      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $35.8m (34.0%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
PCR - Processed rice           19035712.0  33065031.8     18.1     17.5
OFD - Food products n.e.c.      8550399.6  15933895.6      8.1      8.4
SGR - Sugar                     2424695.5   5908252.9      2.3      3.1
MIL - Dairy products            1829426.7    924610.3      1.7      0.5
OMT - Meat products n.e.c.      1820704.7    954507.0      1.7      0.5
VOL - Vegetable oils and fats   1380960.0   1288779.3      1.3      0.7
V_F - Vegetables, fruit, nuts    510189.1    893166.0      0.5      0.5
PDR - Paddy rice                  76561.9    132988.0      0.1      0.1
CMT - Bovine meat prods           73346.0     36346.8      0.1      0.0
OCR - Crops n.e.c.                50976.7     14319.8      0.0      0.0
GRO - Cereal grains n.e.c.        11451.8     27003.4      0.0      0.0

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 42.3% 
GTAP 34.0% 
EORA 41.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46381/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,624,Guinea-Bissau,686,Senegal,Import,Sea,10515,"010515 | Poultry; live, guine",0105,"0105 | Poultry; live, fowls",01,01 | Animals; live,NaN,NaN,NaN,NaN,NaN,0.35600,NaN,0.35600,NaN,0.00,0.30,0.158384,398.0,247.305658,0.039169,['Container'],['Container'],0.000201,0.000200,0.000200,0.000200,0.000201,0.000201,0.00021,0.000373,0.000231,0.000215,0.000216,0.000221,0.000206,0.000365,0.000244,0.000223,0.000218,0.000228,0.000202,0.000397,0.000302,0.000265,0.000245,0.000238,0.000198,0.000422,0.000347,0.000325,0.000262,0.000255,0.000194,0.000428,0.000410,0.000413,0.000284,0.000268,0.000191,0.000419,0.000401,0.000439,0.000326,0.000273,0.000187,0.000374,0.000390,0.000408,0.000305,0.000276,0.000183,0.000343,0.000349,0.000433,0.000358,0.000278,nan,NaN
1,1,624,Guinea-Bissau,620,Portugal,Import,Sea,20120,020120 | Meat; of bovine anim,0201,0201 | Meat of bovine anima,02,02 | Meat and edible meat,21111,"Meat of bovine animals, fresh or chilled",Agriculture,19,CMT - Bovine meat prods,255.03213,NaN,255.03213,kg,138.55,138.55,38.280323,3360.0,2087.806560,79.921909,['Container'],['Container'],0.409718,0.408994,0.409065,0.409065,0.409389,0.409389,0.42865,0.760501,0.472236,0.439053,0.440256,0.451163,0.419695,0.744461,0.497408,0.455839,0.445314,0.465573,0.411841,0.810599,0.617076,0.539751,0.499269,0.485306,0.404132,0.861543,0.708851,0.663572,0.534457,0.520180,0.396569,0.873809,0.835604,0.843249,0.578818,0.546455,0.388854,0.855792,0.818965,0.896738,0.665474,0.556494,0.381586,0.762559,0.795467,0.833321,0.622913,0.563071,0.374175,0.700556,0.711868,0.883889,0.731102,0.566825,21111,Meat and Prep


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $35.1m of $105.1m (33.4%)
Volume: 59.1 of 188.7 million kilograms (31.3%)
Value-to-Weight: 0.59 from 0.56 US$ per kg (+6.8%)
Average distance to market (volume-weighted): 5202.7 from 2994.3 nm (+73.8%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    195  26246539.6  48186021.8     25.0     25.5
1      Sugar and Honey     47   2581965.1   6098148.5      2.5      3.2
2                Dairy     73   1887993.2   1002802.1      1.8      0.5
3        Meat and Prep     46   1724975.3    933474.9      1.6      0.5
4        Fats and Oils     43   1374858.7   1284657.7      1.3      0.7
5           Vegetables    130   1259577.1   1510005.9      1.2      0.8
6               Fruits     35     50058.5     44228.1      0.0      0.0
7  Unprocessed Cereals     10     16032.5     42891.1      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1171832.8,2051865.2,1285099.8,1197648.6,1200426.7,1231209.0,1086753.2,2357949.7,2254096.2,2279217.5,1572085.5,1483629.7,1027365.4,1906927.6,1935365.3,2376627.4,1965644.3,1537705.2
1,Fats and Oils,2549705.4,4116757.0,2716118.9,2549417.7,2556358.8,2647201.1,2396440.1,4623793.4,4465520.0,4623419.4,3258163.6,3088662.7,2289457.0,3857251.3,3953229.1,4622223.5,3830965.0,3181698.4
2,Fruits,34250.1,60297.3,37627.7,35029.6,35116.6,36013.5,31731.2,69261.6,66126.6,66827.7,46068.4,43502.2,29973.0,56069.0,56907.1,69957.9,57748.3,45113.6
3,Meat and Prep,1321179.2,2344003.4,1455518.3,1353242.2,1356949.1,1390569.4,1222297.5,2693240.1,2575487.4,2599048.9,1784024.8,1684276.2,1153275.6,2159242.6,2194110.0,2724310.3,2253392.7,1747058.5
4,Processed Cereals,106013247.5,167066462.1,112088900.7,106116497.8,106406023.5,109811316.3,100070632.2,200094282.9,190586814.4,196785349.5,139190181.5,126706779.4,95922534.3,163386821.5,165601206.9,191274819.3,161896523.9,130081607.8
5,Sugar and Honey,4541547.9,7978279.5,4985617.0,4642479.1,4655172.1,4773410.5,4209151.7,9197244.1,8792274.1,8889450.2,6117441.8,5757901.2,3977130.6,7385350.8,7502940.6,9261636.0,7674387.9,5967468.6
6,Unprocessed Cereals,47070.6,81040.8,51318.8,47966.6,48059.2,49343.8,43785.6,93288.5,89137.6,90379.5,62697.4,59065.3,41492.6,76118.0,77157.8,93509.1,77398.0,61148.3
7,Vegetables,2041174.0,3475648.2,2217051.7,2076563.8,2079579.7,2136702.5,1902497.1,3995988.8,3817060.2,3876275.2,2701119.2,2547085.5,1805655.9,3291486.3,3332194.1,3995528.0,3304008.1,2635422.4


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.47,0.06,0.01,0.02,0.03,0.67,0.62,0.63,0.26,0.21,0.47,0.48,0.71,0.50,0.27
1,Fats and Oils,1.14,0.12,-0.00,0.00,0.07,1.62,1.50,1.62,0.63,0.50,1.14,1.21,1.70,1.12,0.65
2,Fruits,0.52,0.07,0.02,0.02,0.04,0.75,0.69,0.70,0.29,0.24,0.52,0.54,0.80,0.55,0.30
3,Meat and Prep,0.59,0.08,0.02,0.02,0.04,0.85,0.78,0.80,0.33,0.27,0.58,0.60,0.91,0.64,0.34
4,Processed Cereals,2.33,0.23,0.00,0.01,0.14,3.81,3.45,3.68,1.49,1.01,2.57,2.65,3.63,2.51,1.30
5,Sugar and Honey,1.33,0.17,0.04,0.04,0.09,1.93,1.78,1.81,0.74,0.60,1.32,1.37,2.05,1.43,0.77
6,Unprocessed Cereals,2.12,0.26,0.06,0.06,0.14,3.09,2.83,2.91,1.18,0.95,2.16,2.22,3.24,2.24,1.23
7,Vegetables,1.14,0.14,0.03,0.03,0.08,1.66,1.52,1.57,0.63,0.51,1.18,1.21,1.74,1.19,0.66
